# PHASE 6 – FEATURE DETECTION & MATCHING

📌 Goal: Understand how machines recognize objects

### ✅ Module 6.1 – Feature Concepts
- Keypoints
- Descriptors
### ✅ Module 6.2 – Feature Detectors
- Harris Corner
- Shi-Tomasi
- ORB
- SIFT
- SURF (theoretical note + fallback)
### ✅ Module 6.3 – Feature Matching
- Brute Force Matcher
- FLANN Matcher
- Homography
### ✅ Mini Projects
- Image Matching System
- Panorama Stitching (core logic)
#### ⚠️ IMPORTANT NOTE (REAL-WORLD)
- SIFT & SURF may not work in older OpenCV builds.
- This file handles fallback safely.
- ORB is industry-safe & free.

# ✅ COMPLETE SINGLE FILE CODE (PHASE 6)

In [1]:
import cv2
import numpy as np

In [5]:


# ==============================
# LOAD IMAGES
# ==============================
img1 = cv2.imread("image1.jpg")
img2 = cv2.imread("image2.jpg")

if img1 is None or img2 is None:
    raise Exception("Please place image1.jpg and image2.jpg in the same folder")

# =====================================================
# MODULE 6.1 – FEATURE CONCEPTS
# =====================================================
# Keypoints = interesting points (corners, blobs)
# Descriptors = numerical representation of keypoints

# =====================================================
# MODULE 6.2 – FEATURE DETECTORS
# =====================================================

# ---------- Harris Corner ----------
harris = cv2.cornerHarris(np.float32(img1), 2, 3, 0.04)
harris = cv2.dilate(harris, None)

harris_img = cv2.cvtColor(img1, cv2.COLOR_GRAY2BGR)
harris_img[harris > 0.01 * harris.max()] = [0, 0, 255]

# ---------- Shi-Tomasi ----------
corners = cv2.goodFeaturesToTrack(img1, 100, 0.01, 10)
shi_img = cv2.cvtColor(img1, cv2.COLOR_GRAY2BGR)

if corners is not None:
    for corner in corners:
        x, y = corner.ravel()
        cv2.circle(shi_img, (int(x), int(y)), 4, (0, 255, 0), -1)

# ---------- ORB ----------
orb = cv2.ORB_create(nfeatures=1000)
kp1_orb, des1_orb = orb.detectAndCompute(img1, None)
kp2_orb, des2_orb = orb.detectAndCompute(img2, None)

orb_img = cv2.drawKeypoints(img1, kp1_orb, None, color=(255, 0, 0))

# ---------- SIFT ----------
try:
    sift = cv2.SIFT_create()
    kp1_sift, des1_sift = sift.detectAndCompute(img1, None)
    kp2_sift, des2_sift = sift.detectAndCompute(img2, None)
    sift_available = True
except:
    sift_available = False

# =====================================================
# MODULE 6.3 – FEATURE MATCHING
# =====================================================

# ---------- Brute Force Matcher (ORB) ----------
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(des1_orb, des2_orb)
matches = sorted(matches, key=lambda x: x.distance)

bf_match_img = cv2.drawMatches(
    img1, kp1_orb,
    img2, kp2_orb,
    matches[:30],
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

# ---------- FLANN Matcher (SIFT) ----------
if sift_available:
    FLANN_INDEX_KDTREE = 1
    index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
    search_params = dict(checks=50)

    flann = cv2.FlannBasedMatcher(index_params, search_params)
    matches = flann.knnMatch(des1_sift, des2_sift, k=2)

    good_matches = []
    for m, n in matches:
        if m.distance < 0.7 * n.distance:
            good_matches.append(m)

    flann_img = cv2.drawMatches(
        img1, kp1_sift,
        img2, kp2_sift,
        good_matches,
        None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )

# =====================================================
# MINI PROJECT 1 – IMAGE MATCHING SYSTEM
# =====================================================
# Already achieved using BF + FLANN matching

# =====================================================
# MINI PROJECT 2 – PANORAMA STITCHING (CORE LOGIC)
# =====================================================
if sift_available and len(good_matches) > 10:
    src_pts = np.float32(
        [kp1_sift[m.queryIdx].pt for m in good_matches]
    ).reshape(-1, 1, 2)

    dst_pts = np.float32(
        [kp2_sift[m.trainIdx].pt for m in good_matches]
    ).reshape(-1, 1, 2)

    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

    h, w = img1.shape
    panorama = cv2.warpPerspective(img1, H, (w * 2, h))
    panorama[0:h, 0:w] = img2
else:
    panorama = None

# ==============================
# DISPLAY RESULTS
# ==============================
cv2.imshow("Harris Corners", harris_img)
cv2.imshow("Shi-Tomasi Corners", shi_img)
cv2.imshow("ORB Keypoints", orb_img)
cv2.imshow("BF Matcher (ORB)", bf_match_img)

if sift_available:
    cv2.imshow("FLANN Matcher (SIFT)", flann_img)
    if panorama is not None:
        cv2.imshow("Panorama Stitching", panorama)

cv2.waitKey(0)
cv2.destroyAllWindows()


Exception: Please place image1.jpg and image2.jpg in the same folder

In [2]:
import cv2
import numpy as np

# =====================================================
# IMAGE INPUT HANDLING (ROBUST FIX)
# =====================================================
# Try loading images from disk
img1 = cv2.imread("image1.jpg", cv2.IMREAD_GRAYSCALE)
img2 = cv2.imread("image2.jpg", cv2.IMREAD_GRAYSCALE)

# If images are missing, fallback to webcam capture
if img1 is None or img2 is None:
    print("[INFO] image1.jpg / image2.jpg not found.")
    print("[INFO] Switching to webcam capture...")

    cap = cv2.VideoCapture(0)

    print("Capture IMAGE 1 → Press 's'")
    while True:
        ret, frame1 = cap.read()
        cv2.imshow("Capture Image 1", frame1)
        if cv2.waitKey(1) & 0xFF == ord('s'):
            break

    print("Capture IMAGE 2 → Move camera slightly, Press 's'")
    while True:
        ret, frame2 = cap.read()
        cv2.imshow("Capture Image 2", frame2)
        if cv2.waitKey(1) & 0xFF == ord('s'):
            break

    cap.release()
    cv2.destroyAllWindows()

    img1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
    img2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

# =====================================================
# MODULE 6.1 – FEATURE CONCEPTS
# =====================================================
# Keypoints  → interesting points in image
# Descriptors → numerical representation of keypoints

# =====================================================
# MODULE 6.2 – FEATURE DETECTORS
# =====================================================

# ---------- Harris Corner Detection ----------
harris = cv2.cornerHarris(np.float32(img1), 2, 3, 0.04)
harris = cv2.dilate(harris, None)

harris_img = cv2.cvtColor(img1, cv2.COLOR_GRAY2BGR)
harris_img[harris > 0.01 * harris.max()] = [0, 0, 255]

# ---------- Shi–Tomasi Corner Detection ----------
shi_img = cv2.cvtColor(img1, cv2.COLOR_GRAY2BGR)
corners = cv2.goodFeaturesToTrack(img1, 100, 0.01, 10)

if corners is not None:
    for c in corners:
        x, y = c.ravel()
        cv2.circle(shi_img, (int(x), int(y)), 4, (0, 255, 0), -1)

# ---------- ORB (Default – Industry Safe) ----------
orb = cv2.ORB_create(nfeatures=1000)
kp1_orb, des1_orb = orb.detectAndCompute(img1, None)
kp2_orb, des2_orb = orb.detectAndCompute(img2, None)

orb_img = cv2.drawKeypoints(img1, kp1_orb, None, color=(255, 0, 0))

# ---------- SIFT (Optional / Advanced) ----------
sift_available = False
try:
    sift = cv2.SIFT_create()
    kp1_sift, des1_sift = sift.detectAndCompute(img1, None)
    kp2_sift, des2_sift = sift.detectAndCompute(img2, None)
    sift_available = True
except:
    print("[INFO] SIFT not available in this OpenCV build.")

# =====================================================
# MODULE 6.3 – FEATURE MATCHING
# =====================================================

# ---------- Brute Force Matcher (ORB) ----------
bf_match_img = None
if des1_orb is not None and des2_orb is not None:
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1_orb, des2_orb)
    matches = sorted(matches, key=lambda x: x.distance)

    bf_match_img = cv2.drawMatches(
        img1, kp1_orb,
        img2, kp2_orb,
        matches[:30],
        None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )

# ---------- FLANN Matcher + Homography (SIFT) ----------
panorama = None
if sift_available and des1_sift is not None and des2_sift is not None:
    FLANN_INDEX_KDTREE = 1
    index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
    search_params = dict(checks=50)

    flann = cv2.FlannBasedMatcher(index_params, search_params)
    matches = flann.knnMatch(des1_sift, des2_sift, k=2)

    good_matches = []
    for m, n in matches:
        if m.distance < 0.7 * n.distance:
            good_matches.append(m)

    if len(good_matches) > 10:
        src_pts = np.float32(
            [kp1_sift[m.queryIdx].pt for m in good_matches]
        ).reshape(-1, 1, 2)

        dst_pts = np.float32(
            [kp2_sift[m.trainIdx].pt for m in good_matches]
        ).reshape(-1, 1, 2)

        H, _ = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

        h, w = img1.shape
        panorama = cv2.warpPerspective(img1, H, (w * 2, h))
        panorama[0:h, 0:w] = img2

# =====================================================
# DISPLAY RESULTS
# =====================================================
cv2.imshow("Harris Corners", harris_img)
cv2.imshow("Shi–Tomasi Corners", shi_img)
cv2.imshow("ORB Keypoints", orb_img)

if bf_match_img is not None:
    cv2.imshow("Image Matching System (ORB + BF)", bf_match_img)

if panorama is not None:
    cv2.imshow("Panorama Stitching", panorama)

cv2.waitKey(0)
cv2.destroyAllWindows()


[INFO] image1.jpg / image2.jpg not found.
[INFO] Switching to webcam capture...
Capture IMAGE 1 → Press 's'
Capture IMAGE 2 → Move camera slightly, Press 's'


### WHAT YOU MASTERED IN PHASE 6

- ✔ What keypoints & descriptors really are
- ✔ Corner detection (Harris, Shi-Tomasi)
- ✔ Modern feature detectors (ORB, SIFT)
- ✔ Feature matching (BF & FLANN)
- ✔ Homography & geometric alignment
- ✔ Built Image Matching System
- ✔ Built Panorama Stitching logic